# Prepare Inference or Training Data

This notebook pulls forward-filled Forex time series data from a previously prepared InfluxDB database and converts it into a Pandas dataframe suitable for downstream analyses, including LSTM training on the time series.

## Import useful libraries

In [ ]:
import os
import datetime
import json
import pprint as pp

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.sql import Window
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType

In [ ]:
from python_tools_and_shortcuts.databases.influxdb.InfluxDbTool import InfluxDbTool
from python_tools_and_shortcuts.time_series_essentials.day_and_week.day_and_week import day_sin, day_cos, week_sin, week_cos

In [ ]:
# temp
import sys
sys.path.append('/home/emily/Desktop/projects/cc_test/Data-Science/Data-Engineering/ETL')

In [ ]:
from forex.etl.config.database_config import INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG, INFLUXDB_BUCKET
from forex.critical_timezone import critical_timezone

In [ ]:
# find a home for this
from granularity_to_seconds import granularity_to_seconds

## User settings (Papermill)

In [ ]:
instrument = 'EUR/USD'
granularity = 'H1'
n_back = 200
lookahead = 4
MA_lookback_list_as_string = '12,30,50'

verbose = True

training_and_testing = True

#output_directory = os.environ['FOREX_HOME'] + '/output/modeling/data'
output_directory = 'output'

inference_rows_to_keep = 1
qa_reload = False

## User settings (intentionally hard-coded)

In [ ]:
columns_partition = ['instrument', 'granularity']
column_timestamp = 'unix_epoch_s'
MA_columns_list = ['volatility', 'return', 'diff_spread_close', 'diff_volume']
columns_base = ['mid_open', 'mid_high', 'mid_low', 'mid_close', 'spread_close', 'volume']

## Define the timestamp range

In [ ]:
seconds_in_period = granularity_to_seconds[granularity]

max_timestamp = int(datetime.datetime.now().timestamp())

if training_and_testing:
    min_timestamp = int(datetime.datetime(2015, 1, 1, 0, 0, 0).timestamp())
else:
    min_timestamp = max_timestamp - (seconds_in_period * 3 * n_back)

In [ ]:
if verbose:
    print(min_timestamp, max_timestamp)

## Declare user-defined functions

We'll want to move these into a separate module later.

In [ ]:
@F.udf(returnType=FloatType())
def get_day_sin(timestamp):
    return float(day_sin(timestamp))

@F.udf(returnType=FloatType())
def get_day_cos(timestamp):
    return float(day_cos(timestamp))

@F.udf(returnType=FloatType())
def get_week_sin(timestamp):
    return float(week_sin(timestamp))

@F.udf(returnType=FloatType())
def get_week_cos(timestamp):
    return float(week_cos(timestamp))

## Define a function for computing the maximum timestamp in a dataframe

In [ ]:
def get_max_timestamp(df, tz = critical_timezone):
    ts = (
        df
        .groupBy('instrument', 'granularity')
        .agg(
            F.max(F.col(column_timestamp)).alias('max_timestamp')
        )
        .take(1)
        [0]
        ['max_timestamp']
    )

    return datetime.datetime.fromtimestamp(ts, tz = tz)

## Initialize useful variables

In [ ]:
MA_lookback_list = [int(q.strip()) for q in MA_lookback_list_as_string.split(',')]

In [ ]:
if verbose:
    print(MA_lookback_list)

In [ ]:
columns_sort = columns_partition.copy()
columns_sort.append(column_timestamp)

## Initialize configuration data structure

This is to save for downstream use.

In [ ]:
config = {
    'instrument' : instrument,
    'granularity' : granularity,
    'n_back' : n_back,
    'lookahead' : lookahead,
    'MA_lookback_list' : MA_lookback_list,
    'verbose' : verbose,
    'training_and_testing' : training_and_testing,
    'inference_rows_to_keep' : inference_rows_to_keep,
    'output_directory' : output_directory,
    'qa_reload' : qa_reload,

    'columns_partition' : columns_partition,
    'column_timestamp' : column_timestamp,
    'columns_sort' : columns_sort,
    'MA_columns_list' : MA_columns_list,
    'columns_base' : columns_base,

    'min_timestamp' : min_timestamp,
    'max_timestamp' : max_timestamp,
}

## Start a Spark session

In [ ]:
conf = (
    SparkConf()
    .setAppName("MyApp")
    .set("spark.executor.memory", "70G")
    .set("spark.driver.memory", "70G")
    .set("spark.driver.maxResultSize", "70G")
)

spark = SparkSession.builder.config(conf = conf).getOrCreate()

## Connect to the database

In [ ]:
ifc = InfluxDbTool(INFLUXDB_URL, INFLUXDB_TOKEN, INFLUXDB_ORG)

## Pull the data

In [ ]:
def pull_database_data(
    measurement_name,
    ifc,
    instrument,
    granularity,
    columns_sort = ['instrument', 'granularity', 'unix_epoch_s'],
    column_timestamp = 'unix_epoch_s',
):
    query = f'''
        start_s = {min_timestamp}
        stop_s = {max_timestamp}
            
        from(bucket: "forex")
          |> range(
              start: time(v: int(v: start_s) * 1000000000),
              stop: time(v: int(v: stop_s) * 1000000000)
            )
          |> filter(fn: (r) => r._measurement == "{measurement_name}")
          |> filter(fn: (r) => r.granularity == "{granularity}")
          |> filter(fn: (r) => r.instrument == "{instrument}")
          |> pivot(
              rowKey: ["_time"],
              columnKey: ["_field"],
              valueColumn: "_value"
          )
          |> drop(columns: ["_start", "_stop", "_measurement"])
        '''
        
    df = ifc.run_flux_query_on_forex_database_and_get_dataframe(query)
    df.sort_values(by = columns_sort, inplace = True)
    return df

In [ ]:
pdf = pull_database_data(
    'forward-filled candlestick',
    ifc,
    instrument,
    granularity,
    columns_sort = columns_sort,
    column_timestamp = column_timestamp,
)
    
del(ifc)

In [ ]:
columns_to_keep_for_now = columns_sort.copy()
columns_to_keep_for_now.extend([x for x in pdf.columns if not x in columns_sort])

df = (
    spark
    .createDataFrame(pdf)
    .select(*columns_to_keep_for_now)
    .orderBy(*columns_sort)
    .repartition(*columns_partition)
)

In [ ]:
if verbose:
    df.show(3)

In [ ]:
if verbose:
    print(df.count())

In [ ]:
columns_to_keep = columns_sort.copy()
columns_to_keep.extend(columns_base)
df = df.select(*columns_to_keep)

In [ ]:
if verbose:
    df.show(3)

## Filter by instrument and granularity

This is an insurance procedure, the data pull should have only returned one (instrument, granularity) pair.

In [ ]:
if verbose:
    (
        df
        .groupBy('instrument', 'granularity')
        .agg(
            F.count(column_timestamp).alias('timestamp_count'),
        )
        .show(3)
    )

In [ ]:
df = (
    df
    .where(F.col('instrument') == instrument)
    .where(F.col('granularity') == granularity)
)

In [ ]:
if verbose:
    (
        df
        .groupBy('instrument', 'granularity')
        .agg(
            F.count(column_timestamp).alias('timestamp_count'),
        )
        .show(3)
    )

## Compute the potential independent variables (features)

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort)

df = (
    df
    .withColumn('day_sin', get_day_sin(F.col(column_timestamp)))
    .withColumn('day_cos', get_day_cos(F.col(column_timestamp)))
    .withColumn('week_sin', get_week_sin(F.col(column_timestamp)))
    .withColumn('week_cos', get_week_cos(F.col(column_timestamp)))

    .withColumn('volatility', F.col('mid_high') - F.col('mid_low'))
    .withColumn('return', F.col('mid_close') - F.col('mid_open'))

    .withColumn('diff_spread_close', F.col('spread_close') - F.lag(F.col('spread_close'), 1).over(windowSpec))

    .withColumn('diff_volume', F.col('volume') - F.lag(F.col('volume'), 1).over(windowSpec))
)

## Compute the potential dependent variables

#### Percent difference on the return over lead period

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort)

df = (
    df
    .withColumn('mid_close_lead', F.lead(F.col('mid_close'), lookahead).over(windowSpec))
    .withColumn('pd_lead', 100. * ( F.col('mid_close_lead') - F.col('mid_close') ) / F.col('mid_close') )
    .drop('mid_close_lead')
)

#### Spread lead periods ahead

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort)

df = (
    df
    .withColumn('spread_close_lead', F.lead(F.col('spread_close'), lookahead).over(windowSpec))
)

#### Volatility from one ahead to lookahead ahead

Includes every time point in between:

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort).rowsBetween(1, lookahead)

df = (
    df
    .withColumn('mid_high_lookahead', F.collect_list(F.col('mid_high')).over(windowSpec))
    .withColumn('mid_low_lookahead', F.collect_list(F.col('mid_low')).over(windowSpec))
    .withColumn('max_high_lookahead', F.array_max(F.col('mid_high_lookahead')))
    .withColumn('min_low_lookahead', F.array_min(F.col('mid_low_lookahead')))
    .withColumn('volatility_lead', F.col('max_high_lookahead') - F.col('min_low_lookahead'))
)

df = df.drop('mid_low_lookahead', 'mid_high_lookahead', 'min_low_lookahead', 'max_high_lookahead')

## Clean up after computing independent and dependent variables

In [ ]:
df = df.drop('mid_open', 'mid_high', 'mid_low', 'mid_close', 'spread_close', 'volume')

if training_and_testing:
    df = df.dropna()

df = df.orderBy(*columns_sort)

In [ ]:
if verbose:
    df.show(3)

## Compute the row number

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort)
df = df.withColumn('row_num', F.row_number().over(windowSpec))

In [ ]:
if verbose:
    df.show(3)

## Compute the moving averages

In [ ]:
for MA_lookback in MA_lookback_list:
    windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort).rowsBetween(1 - MA_lookback, 0)
    for column_name in MA_columns_list:
        df = df.withColumn(
            column_name + '_MA_' + str(MA_lookback),
            F.avg(F.col(column_name)).over(windowSpec)
        )
df = df.orderBy(*columns_sort)

## Remove incompletely computed rows

In [ ]:
df = df.where(F.col('row_num') >= max(MA_lookback_list))

if training_and_testing:
    count = df.count()
    df = df.where(F.col('row_num') <= count - lookahead)

df = df.orderBy(*columns_sort).drop('row_num')

In [ ]:
if verbose:
    df.show(3)

## Focus on only the columns we care about

In [ ]:
columns_y = ['pd_lead', 'spread_close_lead', 'volatility_lead']
to_select_sans_x = ['instrument', 'granularity', 'unix_epoch_s']
to_select_sans_x.extend(columns_y)

columns_x = [c for c in df.columns if not c in to_select_sans_x]
to_select = to_select_sans_x.copy(); to_select.extend(columns_x)

df = df.orderBy(*columns_sort).select(*to_select)

In [ ]:
if verbose:
    df.show(3)

In [ ]:
if verbose:
    print(df.count())

In [ ]:
if verbose:
    print(len(columns_x))

## Save the columns information to the configuration

In [ ]:
config['columns_x'] = columns_x
config['columns_y'] = columns_y

## Convert to lists

In [ ]:
windowSpec = Window.partitionBy(*columns_partition).orderBy(*columns_sort).rowsBetween(1 - n_back, 0)

df_lists = df.repartition(*columns_partition)

for column_name in columns_x:
    df_lists = (
        df_lists
        .withColumn(column_name, F.collect_list(F.col(column_name)).over(windowSpec))
    )

df_lists = df_lists.repartition(*columns_partition)

df_lists = df_lists.cache() 

df_lists = (
    df_lists
    .where(F.array_size(F.col(columns_x[0])) == F.lit(n_back))
    .orderBy(*columns_sort)
)

In [ ]:
if verbose:
    df_lists.show(3)

In [ ]:
if verbose:
    print(df_lists.count())

## Save the "lists" version of the dataframe

In [ ]:
if training_and_testing:
    output_filename = output_directory + '/df_training_and_testing_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'
else:
    output_filename = output_directory + '/df_to_infer_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'

(
    df_lists
    .repartition(*columns_partition)
    .orderBy(*columns_sort)
    .write
    .mode('overwrite')
    .parquet(output_filename)
)

## Save the "non-lists" version of the dataframe

In [ ]:
if training_and_testing:
    output_filename = output_directory + '/df_training_and_testing_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'
else:
    output_filename = output_directory + '/df_to_infer_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.parquet'

(
    df
    .repartition(*columns_partition)
    .orderBy(*columns_sort)
    .write
    .mode('overwrite')
    .parquet(output_filename)
)

## Save configuration

In [ ]:
if verbose:
    pp.pprint(config)

In [ ]:
if training_and_testing:
    output_filename = output_directory + '/config_training_and_testing_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.json'
else:
    output_filename = output_directory + '/config_to_infer_time_series__' + instrument.replace('/', '_') + '__' + granularity + '__' + str(n_back) + '__' + str(lookahead) + '.json'

to_json = json.dumps(config, indent = 2)
with open(output_filename, 'w') as f:
    f.write(to_json + '\n')

## QA #1

In [ ]:
if verbose:
    pdf = df_lists.orderBy(*columns_sort).toPandas()
    pdf['array_length'] = [len(x) for x in pdf['day_sin']]
    print(pdf['array_length'].unique())

## QA #2

In [ ]:
if qa_reload:
    if training_and_testing:
        output_filename = output_directory + '/df_training_and_testing_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '.parquet'
    else:
        output_filename = output_directory + '/df_to_infer_non_time_series__' + instrument.replace('/', '_') + '__' + granularity + '.parquet'

    df_non_time_series = spark.read.parquet(output_filename)

In [ ]:
if qa_reload:
    if verbose:
        print(df_non_time_series.count())

In [ ]:
if qa_reload:
    if verbose:
        df_non_time_series.show(3)

In [ ]:
if qa_reload:
    if training_and_testing:
        output_filename = output_directory + '/df_training_and_testing_time_series__' + instrument.replace('/', '_') + '__' + granularity + '.parquet'
    else:
        output_filename = output_directory + '/df_to_infer_time_series__' + instrument.replace('/', '_') + '__' + granularity + '.parquet'

    df_time_series = spark.read.parquet(output_filename)

In [ ]:
if qa_reload:
    if verbose:
        print(df_time_series.count())

In [ ]:
if qa_reload:
    if verbose:
        df_time_series.show(3)

## QA #3 (Check final timestamps)

In [ ]:
print(datetime.datetime.fromtimestamp(min_timestamp, tz = critical_timezone))

In [ ]:
print(get_max_timestamp(df))
print(get_max_timestamp(df_lists))

## Close Spark connection

In [ ]:
spark.stop()